# STA/LTA replica vs ground-truth labels

Compares picks from the STA/LTA replica (`my_eq_nonoise_full.csv`) with ground-truth earthquake labels (`eq_labels.csv`).

Labels use a different `file_name` format (e.g. `FN.KLF.00.BHZ | 2025-01-01T09:22:02Z - 2025-01-01T09:23:02Z`); we derive a **window key** (station + time window) so we can align replica and labels per (station, 60s window) and then match P/S times within a tolerance.

In [7]:
import pandas as pd
import numpy as np
import re
from datetime import timedelta

pd.options.display.max_rows = 20
pd.options.display.width = 140

In [8]:
LABELS_CSV = "eq_labels.csv"
REPLICA_CSV = "my_eq_nonoise_full.csv"
TIME_TOLERANCE_SEC = 2.0

## 1. Load and parse labels

Labels `file_name` format: `NET.STA.LOC.CHAN | START - END` (e.g. `FN.KLF.00.BHZ | 2025-01-01T09:22:02.766970Z - 2025-01-01T09:23:02.766970Z`). We extract **station** (STA) and the time **window** (START, END rounded to whole seconds) to build a key that can match the replica's window_key.

In [9]:
def parse_label_file_name(fname: str):
    """Parse label file_name into station and window key segment.

    Example: 'FN.KLF.00.BHZ | 2025-01-01T09:22:02.766970Z - 2025-01-01T09:23:02.766970Z'
    -> station='KLF', window_str='20250101T092202Z__20250101T092302Z'
    """
    fname = str(fname).strip()
    if " | " not in fname or " - " not in fname:
        return None, None
    left, right = fname.split(" | ", 1)
    parts = left.strip().split(".")
    if len(parts) < 2:
        return None, None
    station = parts[1]
    try:
        start_str, end_str = [s.strip() for s in right.split(" - ", 1)]
        start_dt = pd.to_datetime(start_str, utc=True)
        end_dt = pd.to_datetime(end_str, utc=True)
        # Round to whole seconds for alignment with replica windows
        start_sec = start_dt.floor("s")
        end_sec = end_dt.floor("s")
        s_fmt = start_sec.strftime("%Y%m%dT%H%M%SZ")
        e_fmt = end_sec.strftime("%Y%m%dT%H%M%SZ")
        window_str = f"{s_fmt}__{e_fmt}"
        return station, f"{station}/{window_str}"
    except Exception:
        return None, None


def load_labels(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["file_name"] = df["file_name"].astype(str).str.strip()
    parsed = df["file_name"].map(parse_label_file_name)
    df["station_parsed"] = [p[0] for p in parsed]
    df["window_key"] = [p[1] for p in parsed]
    df["p_arrival_time"] = pd.to_datetime(df["p_arrival_time"], errors="coerce")
    df["s_arrival_time"] = pd.to_datetime(df["s_arrival_time"], errors="coerce")
    return df.dropna(subset=["window_key"])


labels = load_labels(LABELS_CSV)
print("Labels rows:", len(labels))
print("Labels unique window_keys:", labels["window_key"].nunique())
labels.head()

Labels rows: 3026
Labels unique window_keys: 2988


,file_name,network,station,instrument_type,station_lat,station_lon,station_elv,p_arrival_time,p_probability,s_arrival_time,s_probability,station_parsed,window_key
0,FN.KLF.00.BHZ | 2025-01-01T09:22:02.766970Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01 09:22:07.766970+00:00,NaN,2025-01-01 09:22:17.997750+00:00,NaN,KLF,KLF/20250101T092202Z__20250101T092302Z
1,FN.RNF.00.BHZ | 2025-01-01T09:22:18.810700Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01 09:22:23.810700+00:00,NaN,2025-01-01 09:22:45.899900+00:00,NaN,RNF,RNF/20250101T092218Z__20250101T092318Z
2,FN.SGF.00.BHZ | 2025-01-01T09:22:12.997750Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01 09:22:17.997750+00:00,NaN,2025-01-01 09:22:36.134150+00:00,NaN,SGF,SGF/20250101T092212Z__20250101T092312Z
3,HE.HEF.00.BHZ | 2025-01-01T09:21:57.651570Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01 09:22:02.651570+00:00,NaN,2025-01-01 09:22:09.394590+00:00,NaN,HEF,HEF/20250101T092157Z__20250101T092257Z
4,FN.KLF.00.BHZ | 2025-01-02T04:42:08.358870Z - ...,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-02 04:42:13.358870+00:00,NaN,2025-01-02 04:42:15.937140+00:00,NaN,KLF,KLF/20250102T044208Z__20250102T044308Z


## 2. Load replica and add window_key

Replica file_name format: `STATION/NET.STA..CHAN__START__END.mseed`. We use the same window_key convention (station + time window) so we can match with labels.

In [10]:
def replica_file_name_to_window_key(fname: str) -> str:
    fname = str(fname).strip()
    if "/" not in fname or "__" not in fname:
        return fname
    station = fname.split("/")[0]
    rest = fname.split("/", 1)[1]
    parts = rest.split("__")
    if len(parts) >= 3:
        window = parts[-2] + "__" + parts[-1].replace(".mseed", "").strip()
        return f"{station}/{window}"
    return fname


def load_replica(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["file_name"] = df["file_name"].astype(str).str.strip()
    df["window_key"] = df["file_name"].map(replica_file_name_to_window_key)
    for col in ["p_arrival_time", "s_arrival_time", "start_dt", "end_dt"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


replica = load_replica(REPLICA_CSV)
print("Replica rows:", len(replica))
print("Replica unique window_keys:", replica["window_key"].nunique())
replica.head()

Replica rows: 13799
Replica unique window_keys: 2738


,file_name,network,station,instrument_type,station_lat,station_lon,station_elv,p_arrival_time,p_probability,s_arrival_time,s_probability,start_dt,end_dt,window_key
0,HEF/HE.HEF..HHN__20250101T092157Z__20250101T09...,HE,HEF,0,0,0,0,2025-01-01 09:22:05.150,1.0,2025-01-01 09:22:05.150,1.000000,2025-01-01 09:21:57,2025-01-01 09:22:57,HEF/20250101T092157Z__20250101T092257Z
1,KLF/FN.KLF..HHN__20250101T092202Z__20250101T09...,FN,KLF,0,0,0,0,2025-01-01 09:22:08.850,1.0,2025-01-01 09:22:09.450,0.595983,2025-01-01 09:22:02,2025-01-01 09:23:02,KLF/20250101T092202Z__20250101T092302Z
2,KLF/FN.KLF..HHN__20250101T092202Z__20250101T09...,FN,KLF,0,0,0,0,2025-01-01 09:22:15.490,1.0,2025-01-01 09:22:15.490,1.000000,2025-01-01 09:22:02,2025-01-01 09:23:02,KLF/20250101T092202Z__20250101T092302Z
3,KLF/FN.KLF..HHN__20250101T092202Z__20250101T09...,FN,KLF,0,0,0,0,2025-01-01 09:22:20.900,1.0,2025-01-01 09:22:20.900,1.000000,2025-01-01 09:22:02,2025-01-01 09:23:02,KLF/20250101T092202Z__20250101T092302Z
4,KLF/FN.KLF..HHN__20250101T092202Z__20250101T09...,FN,KLF,0,0,0,0,2025-01-01 09:23:02.950,1.0,2025-01-01 09:23:02.950,1.000000,2025-01-01 09:22:02,2025-01-01 09:23:02,KLF/20250101T092202Z__20250101T092302Z


## 3. Align windows

Label and replica window keys may not be identical (e.g. label 09:22:02.766 -> 09:22:02, replica 09:22:02 exact). We compare only on **windows that appear in the labels** (ground truth), and for each such window we take replica rows with the **same** window_key. If labels use a slightly different second (e.g. 09:22:03 vs 09:22:02), we could add a fuzzy match; for simplicity we require exact window_key match, so label windows must round to the same second as replica. If needed, we can relax this by matching on (station, overlapping time range) in a later step.

In [11]:
label_keys = set(labels["window_key"])
replica_keys = set(replica["window_key"])
common_keys = sorted(label_keys & replica_keys)
labels_only = label_keys - replica_keys
replica_only = replica_keys - label_keys

print("Windows in labels:", len(label_keys))
print("Windows in replica:", len(replica_keys))
print("Windows in both (exact key match):", len(common_keys))
print("In labels only:", len(labels_only))
print("In replica only:", len(replica_only))
if labels_only and len(labels_only) <= 5:
    print("Example labels-only keys:", list(labels_only)[:5])
if common_keys:
    print("Example common key:", common_keys[0])

Windows in labels: 2988
Windows in replica: 2738
Windows in both (exact key match): 2738
In labels only: 250
In replica only: 0
Example common key: AAL/20250113T034125Z__20250113T034225Z


## 4. Time-based matching and metrics

For each window in **common_keys**, we collect all P and S times from labels and from replica, then greedily match them within `TIME_TOLERANCE_SEC`. We report:

- **Recall (replica vs labels)**: Of all label P (or S) picks, what fraction did the replica also make within the tolerance?
- **Precision (replica vs labels)**: Of all replica P (or S) picks in label windows, what fraction match a label pick within the tolerance?

In [12]:
def _to_naive_utc(ts):
    t = pd.Timestamp(ts)
    if t.tzinfo is not None:
        t = t.tz_convert("UTC").tz_localize(None)
    return t

def match_times(t1: pd.Series, t2: pd.Series, tol_sec: float) -> int:
    a = sorted([_to_naive_utc(x) for x in t1.dropna()])
    b = sorted([_to_naive_utc(x) for x in t2.dropna()])
    i = j = 0
    matches = 0
    tol = timedelta(seconds=tol_sec)
    while i < len(a) and j < len(b):
        dt = a[i] - b[j]
        if abs(dt) <= tol:
            matches += 1
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return matches


def compute_replica_vs_labels_stats(labels: pd.DataFrame, replica: pd.DataFrame, common_keys: list, tol_sec: float):
    total_gt_p = total_gt_s = 0
    total_rep_p = total_rep_s = 0
    matched_p = matched_s = 0
    rows = []

    for key in common_keys:
        lab = labels[labels["window_key"] == key]
        rep = replica[replica["window_key"] == key]

        gt_p = lab["p_arrival_time"].dropna()
        rep_p = rep["p_arrival_time"].dropna()
        n_gt_p = len(gt_p)
        n_rep_p = len(rep_p)
        n_m_p = match_times(gt_p, rep_p, tol_sec) if n_gt_p and n_rep_p else 0

        gt_s = lab["s_arrival_time"].dropna()
        rep_s = rep["s_arrival_time"].dropna()
        n_gt_s = len(gt_s)
        n_rep_s = len(rep_s)
        n_m_s = match_times(gt_s, rep_s, tol_sec) if n_gt_s and n_rep_s else 0

        total_gt_p += n_gt_p
        total_rep_p += n_rep_p
        matched_p += n_m_p
        total_gt_s += n_gt_s
        total_rep_s += n_rep_s
        matched_s += n_m_s

        rows.append({
            "window_key": key,
            "gt_P": n_gt_p,
            "rep_P": n_rep_p,
            "matched_P": n_m_p,
            "gt_S": n_gt_s,
            "rep_S": n_rep_s,
            "matched_S": n_m_s,
        })

    stats_df = pd.DataFrame(rows)
    overall = {
        "total_gt_P": total_gt_p,
        "total_rep_P": total_rep_p,
        "matched_P": matched_p,
        "P_recall_replica_vs_labels": matched_p / total_gt_p if total_gt_p else np.nan,
        "P_precision_replica_vs_labels": matched_p / total_rep_p if total_rep_p else np.nan,
        "total_gt_S": total_gt_s,
        "total_rep_S": total_rep_s,
        "matched_S": matched_s,
        "S_recall_replica_vs_labels": matched_s / total_gt_s if total_gt_s else np.nan,
        "S_precision_replica_vs_labels": matched_s / total_rep_s if total_rep_s else np.nan,
    }
    return stats_df, overall


stats_df, overall = compute_replica_vs_labels_stats(labels, replica, common_keys, TIME_TOLERANCE_SEC)
print("Overall (only windows that appear in both; tolerance =", TIME_TOLERANCE_SEC, "s):")
for k, v in overall.items():
    print(f"  {k}: {v}")
overall

Overall (only windows that appear in both; tolerance = 2.0 s):
  total_gt_P: 1953
  total_rep_P: 13799
  matched_P: 1024
  P_recall_replica_vs_labels: 0.5243215565796211
  P_precision_replica_vs_labels: 0.07420827596202624
  total_gt_S: 2715
  total_rep_S: 13799
  matched_S: 880
  S_recall_replica_vs_labels: 0.3241252302025783
  S_precision_replica_vs_labels: 0.0637727371548663


{'total_gt_P': 1953,
 'total_rep_P': 13799,
 'matched_P': 1024,
 'P_recall_replica_vs_labels': 0.5243215565796211,
 'P_precision_replica_vs_labels': 0.07420827596202624,
 'total_gt_S': 2715,
 'total_rep_S': 13799,
 'matched_S': 880,
 'S_recall_replica_vs_labels': 0.3241252302025783,
 'S_precision_replica_vs_labels': 0.0637727371548663}

In [13]:
print("Summary:")
print("  P: Of all ground-truth P picks in matched windows, {:.1%} were also detected by the replica (recall).".format(overall["P_recall_replica_vs_labels"]))
print("  P: Of all replica P picks in those windows, {:.1%} matched a ground-truth P (precision).".format(overall["P_precision_replica_vs_labels"]))
print("  S: Recall = {:.1%}, Precision = {:.1%}.".format(overall["S_recall_replica_vs_labels"], overall["S_precision_replica_vs_labels"]))

Summary:
  P: Of all ground-truth P picks in matched windows, 52.4% were also detected by the replica (recall).
  P: Of all replica P picks in those windows, 7.4% matched a ground-truth P (precision).
  S: Recall = 32.4%, Precision = 6.4%.


In [14]:
print("Per-window stats (first 20):")
display(stats_df.head(20))

print("Windows where ground truth has P but replica has no matching P:")
miss = stats_df[(stats_df["gt_P"] > 0) & (stats_df["matched_P"] == 0)]
display(miss.head(15))

Per-window stats (first 20):


,window_key,gt_P,rep_P,matched_P,gt_S,rep_S,matched_S
0,AAL/20250113T034125Z__20250113T034225Z,1,1,0,1,1,1
1,AAL/20250131T071950Z__20250131T072050Z,1,2,1,1,2,1
2,AAL/20250215T204007Z__20250215T204107Z,1,2,1,1,2,1
3,AAL/20250219T151720Z__20250219T151820Z,1,4,1,1,4,0
4,AAL/20250320T013343Z__20250320T013443Z,1,2,1,1,2,1
5,AAL/20250326T050629Z__20250326T050729Z,1,2,1,1,2,1
6,AAL/20250520T011318Z__20250520T011418Z,0,2,0,1,2,0
7,AAL/20250525T125151Z__20250525T125251Z,1,2,1,1,2,0
8,AAL/20250618T163553Z__20250618T163653Z,1,2,1,1,2,1
9,AAL/20250626T014110Z__20250626T014210Z,0,1,0,1,1,0


Windows where ground truth has P but replica has no matching P:


,window_key,gt_P,rep_P,matched_P,gt_S,rep_S,matched_S
0,AAL/20250113T034125Z__20250113T034225Z,1,1,0,1,1,1
12,AAL/20250705T153402Z__20250705T153502Z,1,1,0,1,1,0
17,AAL/20250818T201333Z__20250818T201433Z,1,2,0,1,2,1
29,AAL/20251120T230921Z__20251120T231021Z,1,2,0,1,2,0
33,ALAJF/20250105T045508Z__20250105T045608Z,1,2,0,1,2,0
34,ALAJF/20250107T041433Z__20250107T041533Z,1,1,0,1,1,0
37,ALAJF/20250330T131427Z__20250330T131527Z,1,1,0,1,1,0
38,ALAJF/20250409T175923Z__20250409T180023Z,1,1,0,1,1,0
40,ALAJF/20250423T183510Z__20250423T183610Z,1,3,0,1,3,0
42,ALAJF/20250512T031158Z__20250512T031258Z,1,1,0,1,1,0
